<a href="https://colab.research.google.com/github/elithecreator06/machine-learning-algorithms/blob/main/knn-lvq-linear-regression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd

from sklearn import datasets
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, mean_squared_error

In [ ]:
print("==== Part 1: KNN ====")

#Load the Iris dataset
iris = datasets.load_iris()

#Features and class labels
X = iris.data
y = iris.target

#Split dataset into 70% training and 30% testing
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size = 0.30,
    random_state = 42
)

#calculate Euclidean distance between two points
def euclidean_distance(x1, x2):
  return np.sqrt(np.sum((x1 - x2) ** 2))

#KNN classifier with k = 1
def knn_predict(X_train, y_train, test_instance):
  distances = []

  #Compute distance from test sample to every training sample
  for i in range(len(X_train)):
    distance = euclidean_distance(test_instance, X_train[i])
    distances.append(distance)

  #Find nearest neighbour
  nearest_index = np.argmin(distances)

  return y_train[nearest_index]

#Generate predictions for the test data
predictions = []

for sample in X_test:
  prediction = knn_predict(X_train, y_train, sample)
  predictions.append(prediction)

#Calculate classification accuracy
accuracy_knn = accuracy_score(y_test, predictions)

print("KNN Accuracy = ", accuracy_knn)

==== Part 1: KNN ====
KNN Accuracy =  1.0


In [ ]:
print("\n==== Part 2: LVQ ====")

#Obtain unique class labels
classes = np.unique(y_train)

#Store prototype vectors (one per class)
codebooks = []

#Initialise codebooks using class means
for c in classes:
  class_sample = X_train[y_train == c]
  mean_vector = np.mean(class_sample, axis = 0)
  codebooks.append(mean_vector)

codebooks = np.array(codebooks)

#Set learning parameters
learning_rate = 0.1
epochs = 100

#Train LVQ model
for epoch in range(epochs):
  for i in range(len(X_train)):
    sample = X_train[i]
    label = y_train[i]

    #Find closet codebook vector
    distances = np.linalg.norm(codebooks - sample, axis = 1)
    winner = np.argmin(distances)

    #Move codebook toward correct samples
    #Move codebook away from incorrect samples
    if winner == label:
      codebooks[winner] = codebooks[winner] + learning_rate * (sample - codebooks[winner])
    else:
      codebooks[winner] = codebooks[winner] - learning_rate * (sample - codebooks[winner])

#Classify testing samples
lvq_predictions = []

for sample in X_test:
  distances = np.linalg.norm(codebooks - sample, axis = 1)
  prediction = np.argmin(distances)
  lvq_predictions.append(prediction)

#Calculate classification accuracy
accuracy_lvq = accuracy_score(y_test, lvq_predictions)

print("LVQ Accuracy = ", accuracy_lvq)


==== Part 2: LVQ ====
LVQ Accuracy =  0.9555555555555556


In [ ]:
print("\n==== Comparison ====")

#Display classification accuracy for both methods
print("KNN Accuracy : ", accuracy_knn)
print("LVQ Accuracy : ", accuracy_lvq)

#Determine which algorithm performed better
if accuracy_knn > accuracy_lvq:
  print("KNN performed better. ")
elif accuracy_lvq > accuracy_knn:
  print("LVQ performed better. ")
else:
  print("Both algorithms performed equally. ")


==== Comparison ====
KNN Accuracy :  1.0
LVQ Accuracy :  0.9555555555555556
KNN performed better. 


In [ ]:
from scipy.sparse import data
print("\n==== Part 3: Linear Regression ====")

#Load Boston Housing dataset
data_url = "http://lib.stat.cmu.edu/datasets/boston"

#Read raw dataset
raw_df = pd.read_csv(
    data_url,
    sep = r"\s+",
    skiprows = 22,
    header = None
)

#Extract feature matrix and target values
data = np.hstack([
    raw_df.values[::2, :],
    raw_df.values[1::2, :2]
])

target = raw_df.values[1::2, 2]

X = data
y = target

#Split into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size = 0.30,
    random_state = 42
)

#Normalise features for gradient descent
X_train = (X_train - X_train.mean(axis = 0)) / X_train.std(axis = 0)
X_test = (X_test - X_test.mean(axis = 0)) / X_test.std(axis = 0)

#Initialise model parameters
m, n = X_train.shape
weights = np.zeros(n)
bias = 0

learning_rate = 0.05
epochs = 1000

#Train model using gradient desent
for epoch in range(epochs):
  predictions = np.dot(X_train, weights) + bias
  error = predictions - y_train

  dw = (1/m) * np.dot(X_train.T, error)
  db = (1/m) * np.sum(error)

  weights = weights - learning_rate * dw
  bias = bias - learning_rate * db

#Predict housing prices on the test data
test_predictions = np.dot(X_test, weights) + bias

#Calculate Mean Squared Errror
mse = mean_squared_error(y_test, test_predictions)

print("Mean Squared Error = ", mse)


==== Part 3: Linear Regression ====
Mean Squared Error =  24.175042678409742
